In [5]:
import sqlite3
import numpy as np
import pandas as pd
import json
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
import requests
import time
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from tqdm import tqdm

In [6]:
def obtener_carreras_estrategias_multiples():
    driver = webdriver.Chrome()
    todas_las_carreras = set()
    
    try:
        driver.get("https://comparacarreras.imco.org.mx")
        wait = WebDriverWait(driver, 10)
        search_input = wait.until(EC.presence_of_element_located((By.ID, "search-input")))
        
        # Letras comunes para activar búsquedas
        letras_prueba = ["", "a", "e", "i", "o", "u", "in", "ad", "ci"]
        
        for letra in letras_prueba:
            try:
                search_input.clear()
                search_input.click()
                time.sleep(1)
                
                if letra:
                    search_input.send_keys(letra)
                else:
                    search_input.send_keys(" ")  # Espacio para mostrar todo
                
                time.sleep(3)
                
                # Extraer resultados
                results_container = driver.find_element(By.ID, "search-results")
                opciones = results_container.find_elements(By.CSS_SELECTOR, "*")
                
                for opcion in opciones:
                    texto = opcion.text.strip()
                    if texto and len(texto) > 3 and not texto.isdigit():
                        todas_las_carreras.add(texto)
                        
                
            except Exception as e:
                print(f"Error con '{letra}': {e}")
                continue
        
        # Convertir a lista ordenada
        carreras_final = sorted(list(todas_las_carreras))
        
        print(f"\n🎯 TOTAL: {len(carreras_final)} carreras encontradas:")
        for i, carrera in enumerate(carreras_final, 1):
            print(f"{i:2d}. {carrera}")
            
        return carreras_final
        
    except Exception as e:
        print(f"❌ Error general: {e}")
        return []
    finally:
        driver.quit()

In [56]:
careers = obtener_carreras_estrategias_multiples()


🎯 TOTAL: 81 carreras encontradas:
 1. Administración de empresas
 2. Administración en el ámbito de la ingeniería y la construcción
 3. Administración pública
 4. Administración turística y de restaurantes
 5. Adquisición de idiomas extranjeros
 6. Arquitectura y urbanismo
 7. Bellas artes
 8. Biología
 9. Bioquímica y biofísica
10. Ciencias computacionales
11. Ciencias de la educación
12. Ciencias políticas
13. Comunicación y periodismo
14. Construcción e ingeniería civil
15. Contabilidad y fiscalización
16. Criminología y criminalística
17. Deportes
18. Derecho
19. Desarrollo de software
20. Diseño curricular y pedagogía
21. Diseño industrial de moda e interiores
22. Diseño y comunicación gráfica y editorial
23. Ecología y ciencias ambientales
24. Economía
25. Electricidad y generación de energía
26. Electrónica automatización y aplicaciones de la mecánica-eléctrica
27. Enfermería general y obstetricia
28. Estomatología y odontología general
29. Finanzas banca y seguros
30. Formació

In [8]:
#lista = []
#for i in range(len(careers)):
#    if "TSU" not in careers[i]:
#        lista.append(careers[i])

In [58]:
careers

['Administración de empresas',
 'Administración en el ámbito de la ingeniería y la construcción',
 'Administración pública',
 'Administración turística y de restaurantes',
 'Adquisición de idiomas extranjeros',
 'Arquitectura y urbanismo',
 'Bellas artes',
 'Biología',
 'Bioquímica y biofísica',
 'Ciencias computacionales',
 'Ciencias de la educación',
 'Ciencias políticas',
 'Comunicación y periodismo',
 'Construcción e ingeniería civil',
 'Contabilidad y fiscalización',
 'Criminología y criminalística',
 'Deportes',
 'Derecho',
 'Desarrollo de software',
 'Diseño curricular y pedagogía',
 'Diseño industrial de moda e interiores',
 'Diseño y comunicación gráfica y editorial',
 'Ecología y ciencias ambientales',
 'Economía',
 'Electricidad y generación de energía',
 'Electrónica automatización y aplicaciones de la mecánica-eléctrica',
 'Enfermería general y obstetricia',
 'Estomatología y odontología general',
 'Finanzas banca y seguros',
 'Formación docente en educación básica nivel p

In [61]:
valor_a_cambiar = "TSU. Informática"
nuevo_valor = "tsu.__informática"

if valor_a_cambiar in careers:
    indice = careers.index(valor_a_cambiar)
    careers[indice] = nuevo_valor
    print(careers[indice])

tsu.__informática


In [62]:
def formatear_lista(lista_original):
    return [texto.lower().replace(" ", "_") for texto in lista_original]

# Ejemplo de uso
lista_formateada = formatear_lista(careers)
print(lista_formateada)

['administración_de_empresas', 'administración_en_el_ámbito_de_la_ingeniería_y_la_construcción', 'administración_pública', 'administración_turística_y_de_restaurantes', 'adquisición_de_idiomas_extranjeros', 'arquitectura_y_urbanismo', 'bellas_artes', 'biología', 'bioquímica_y_biofísica', 'ciencias_computacionales', 'ciencias_de_la_educación', 'ciencias_políticas', 'comunicación_y_periodismo', 'construcción_e_ingeniería_civil', 'contabilidad_y_fiscalización', 'criminología_y_criminalística', 'deportes', 'derecho', 'desarrollo_de_software', 'diseño_curricular_y_pedagogía', 'diseño_industrial_de_moda_e_interiores', 'diseño_y_comunicación_gráfica_y_editorial', 'ecología_y_ciencias_ambientales', 'economía', 'electricidad_y_generación_de_energía', 'electrónica_automatización_y_aplicaciones_de_la_mecánica-eléctrica', 'enfermería_general_y_obstetricia', 'estomatología_y_odontología_general', 'finanzas_banca_y_seguros', 'formación_docente_en_educación_básica_nivel_preescolar', 'formación_docent

In [63]:
def getStatistics(careers):
    filas = []
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Ejecutar en segundo plano
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome(options=chrome_options)
    
    try:
        for i in tqdm(range(len(careers))):
            url = f"https://comparacarreras.imco.org.mx/{careers[i]}"
            print(url)
            driver.get(url)
            time.sleep(5)
            page_source = driver.page_source
            total_students = driver.find_element(By.XPATH, '//*[@id="total-students"]/h3').text
            new_graduates_number = driver.find_element(By.ID, "new-graduates-number").text
            
            public_cost = driver.find_element(By.ID, "public-cost").text
            private_cost = driver.find_element(By.ID, "private-cost").text
            public_quality_rating = driver.find_element(By.ID, "public-quality-rating").text
            private_quality_rating = driver.find_element(By.ID, "private-quality-rating").text

            occupation_rate = driver.find_element(By.ID, "occupation-rate").text
            unemployment_rate = driver.find_element(By.ID, "unemployment-rate").text
            informality_rate = driver.find_element(By.ID, "informality-rate").text
            quality_employment_probability = driver.find_element(By.ID, "quality-employment-probability").text

            average_salary = driver.find_element(By.ID, "average-salary").text
            career_rank = driver.find_element(By.ID, "career-rank").text

            women_salary = driver.find_element(By.ID, "women-salary").text
            men_salary = driver.find_element(By.ID, "men-salary").text
            under_30_salary = driver.find_element(By.ID, "under-30-salary").text
            over_30_salary = driver.find_element(By.ID, "over-30-salary").text
            formal_salary = driver.find_element(By.ID, "formal-salary").text
            informal_salary = driver.find_element(By.ID, "informal-salary").text

            postgrad_percentage = driver.find_element(By.ID, "postgrad-percentage").text
            postgrad_salary = driver.find_element(By.ID, "postgrad-salary").text
            salary_increase = driver.find_element(By.ID, "salary-increase").text

            filas.append({
                "id":i,
                "career": careers[i],
                "total_students":total_students,
                "new_graduates_number":new_graduates_number,
                "public_cost": public_cost,
                "private_cost": private_cost,
                "public_quality_rating": public_quality_rating,
                "private_quality_rating": private_quality_rating,
                "occupation_rate": occupation_rate,
                "unemployment_rate": unemployment_rate,
                "informality_rate": informality_rate,
                "quality_employment_probability": quality_employment_probability,
                "average_salary": average_salary,
                "career_rank": career_rank,
                "women_salary": women_salary,
                "men_salary": men_salary,
                "under_30_salary": under_30_salary,
                "over_30_salary": over_30_salary,
                "formal_salary": formal_salary,
                "informal_salary": informal_salary,
                "postgrad_percentage": postgrad_percentage,
                "postgrad_salary": postgrad_salary,
                "salary_increase": salary_increase
            })
    finally:
        driver.quit()
        print("Completado")
    return filas

In [64]:
data = getStatistics(lista_formateada)

  0%|                                                                                           | 0/81 [00:00<?, ?it/s]

https://comparacarreras.imco.org.mx/administración_de_empresas


  1%|█                                                                                  | 1/81 [00:06<09:05,  6.82s/it]

https://comparacarreras.imco.org.mx/administración_en_el_ámbito_de_la_ingeniería_y_la_construcción


  2%|██                                                                                 | 2/81 [00:12<08:11,  6.22s/it]

...

https://comparacarreras.imco.org.mx/vehículos_barcos_y_aeronaves_motorizadas


 99%|████████████████████████████████████████████████████████████████████████████████▉ | 80/81 [07:43<00:05,  5.80s/it]

https://comparacarreras.imco.org.mx/veterinaria


100%|██████████████████████████████████████████████████████████████████████████████████| 81/81 [07:49<00:00,  5.79s/it]


Completado


In [65]:
df = pd.json_normalize(data)

In [66]:
df

,id,career,total_students,new_graduates_number,public_cost,private_cost,public_quality_rating,private_quality_rating,occupation_rate,unemployment_rate,...,career_rank,women_salary,men_salary,under_30_salary,over_30_salary,formal_salary,informal_salary,postgrad_percentage,postgrad_salary,salary_increase
0,0,administración_de_empresas,1.3M,47.537,$50.156,$228.756,Buena,Buena,96.8%,3.3%,...,18°,$19.561,$25.207,$16.723,$24.358,$23.740,$16.242,8.5%,$43.317,"236,71%"
1,1,administración_en_el_ámbito_de_la_ingeniería_y...,79.106,3539,-,-,-,-,94.7%,5.3%,...,-°,$-,$-,$-,$-,$-,$-,6.0%,$-,-%
2,2,administración_pública,48.254,1938,-,-,-,-,98.1%,1.9%,...,-°,$-,$-,$-,$-,$-,$-,49.6%,$-,-%
3,3,administración_turística_y_de_restaurantes,61.850,6029,$44.415,$451.397,Insegura,Muy Insegura,93.9%,6.1%,...,32°,$16.651,$20.528,$14.702,$19.567,$19.757,$12.040,7.4%,$26.889,"174,49%"
4,4,adquisición_de_idiomas_extranjeros,77.839,3083,$47.970,$169.694,Muy Insegura,Muy Insegura,94.1%,5.9%,...,40°,$16.595,$19.242,$14.089,$19.676,$18.491,$13.691,3.0%,$27.178,"181,3%"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,76,tecnología_para_la_protección_del_medio_ambiente,34.168,3287,-,-,-,-,94.2%,5.8%,...,-°,$-,$-,$-,$-,$-,$-,8.5%,$-,-%
77,77,terapia_y_rehabilitación,66.243,8890,-,-,-,-,96.8%,3.2%,...,-°,$-,$-,$-,$-,$-,$-,3.7%,$-,-%
78,78,trabajo_y_atención_social,157.770,7898,$33.790,$88.790,Buena,Excelente,97.2%,2.8%,...,43°,$16.935,$17.447,$13.446,$18.335,$18.159,$12.450,3.6%,$27.548,"188,47%"
79,79,vehículos_barcos_y_aeronaves_motorizadas,71.470,6150,-,-,-,-,97.3%,2.7%,...,-°,$-,$-,$-,$-,$-,$-,1.2%,$-,-%


In [67]:
import pandas as pd
import numpy as np
import re

In [68]:
df.isnull().sum()

id                                0
career                            0
total_students                    0
new_graduates_number              0
public_cost                       0
private_cost                      0
public_quality_rating             0
private_quality_rating            0
occupation_rate                   0
unemployment_rate                 0
informality_rate                  0
quality_employment_probability    0
average_salary                    0
career_rank                       0
women_salary                      0
men_salary                        0
under_30_salary                   0
over_30_salary                    0
formal_salary                     0
informal_salary                   0
postgrad_percentage               0
postgrad_salary                   0
salary_increase                   0
dtype: int64

In [69]:
df.dtypes

id                                 int64
career                            object
total_students                    object
new_graduates_number              object
public_cost                       object
private_cost                      object
public_quality_rating             object
private_quality_rating            object
occupation_rate                   object
unemployment_rate                 object
informality_rate                  object
quality_employment_probability    object
average_salary                    object
career_rank                       object
women_salary                      object
men_salary                        object
under_30_salary                   object
over_30_salary                    object
formal_salary                     object
informal_salary                   object
postgrad_percentage               object
postgrad_salary                   object
salary_increase                   object
dtype: object

In [70]:
def fix_number(x):
    x = str(x)
    if "-" == x:
        return 0
    if "M" in x:
        print("Millón")
        x = float(x.replace('M',''))
        x = x * 1000000
        return int(x)
    if x.endswith(".0"):
        parts = x.split('.')
        print(parts[0], flush=True)
        return int(parts[0])
    else:
        return int(float(x.replace('.', '')))

In [71]:
df['new_graduates_number'] = df['new_graduates_number'].apply(fix_number)

In [72]:
df['total_students'] = df['total_students'].apply(fix_number)

Millón
Millón
Millón
Millón


In [73]:
def remove_money_simbol(x):
    if "$" in x:
        x = x.replace('$','')
        return x
    if "-" == x:
        return 0

In [74]:
cols_with_dollar = df.columns[df.astype(str).apply(lambda col: col.str.contains('\$').any())]

In [75]:
cols_with_dollar

Index(['public_cost', 'private_cost', 'average_salary', 'women_salary',
       'men_salary', 'under_30_salary', 'over_30_salary', 'formal_salary',
       'informal_salary', 'postgrad_salary'],
      dtype='object')

In [76]:
for col in cols_with_dollar:
    df[col] = df[col].apply(remove_money_simbol)
    df[col] = df[col].apply(fix_number)

Millón


In [77]:
df

,id,career,total_students,new_graduates_number,public_cost,private_cost,public_quality_rating,private_quality_rating,occupation_rate,unemployment_rate,...,career_rank,women_salary,men_salary,under_30_salary,over_30_salary,formal_salary,informal_salary,postgrad_percentage,postgrad_salary,salary_increase
0,0,administración_de_empresas,1300000,47537,50156,228756,Buena,Buena,96.8%,3.3%,...,18°,19561,25207,16723,24358,23740,16242,8.5%,43317,"236,71%"
1,1,administración_en_el_ámbito_de_la_ingeniería_y...,79106,3539,0,0,-,-,94.7%,5.3%,...,-°,0,0,0,0,0,0,6.0%,0,-%
2,2,administración_pública,48254,1938,0,0,-,-,98.1%,1.9%,...,-°,0,0,0,0,0,0,49.6%,0,-%
3,3,administración_turística_y_de_restaurantes,61850,6029,44415,451397,Insegura,Muy Insegura,93.9%,6.1%,...,32°,16651,20528,14702,19567,19757,12040,7.4%,26889,"174,49%"
4,4,adquisición_de_idiomas_extranjeros,77839,3083,47970,169694,Muy Insegura,Muy Insegura,94.1%,5.9%,...,40°,16595,19242,14089,19676,18491,13691,3.0%,27178,"181,3%"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,76,tecnología_para_la_protección_del_medio_ambiente,34168,3287,0,0,-,-,94.2%,5.8%,...,-°,0,0,0,0,0,0,8.5%,0,-%
77,77,terapia_y_rehabilitación,66243,8890,0,0,-,-,96.8%,3.2%,...,-°,0,0,0,0,0,0,3.7%,0,-%
78,78,trabajo_y_atención_social,157770,7898,33790,88790,Buena,Excelente,97.2%,2.8%,...,43°,16935,17447,13446,18335,18159,12450,3.6%,27548,"188,47%"
79,79,vehículos_barcos_y_aeronaves_motorizadas,71470,6150,0,0,-,-,97.3%,2.7%,...,-°,0,0,0,0,0,0,1.2%,0,-%


In [78]:
def remove_rank_signal(x):
    x = str(x).strip()  
    
    if "°" in x:
        x = x.replace('°', '').strip()
        if x == "-" or x == "": 
            return None
        return int(x)
    
    return x

In [79]:
df['career_rank'] = df['career_rank'].apply(remove_rank_signal)

In [80]:
def remove_percentage_signal(x):
    x = str(x).strip()  
    
    if "%" in x:
        x = x.replace('%', '').strip()
        if x == "-" or x == "": 
            return np.nan
        return float(x)
    
    return x

In [81]:
df['postgrad_percentage'] = df['postgrad_percentage'].apply(remove_percentage_signal)

In [82]:
def remove_(x):
    if "-" == x:
        return np.nan
    else: 
        return x

In [83]:
df['postgrad_percentage'] = df['postgrad_percentage'].apply(remove_)

In [84]:
def clean_percentage(x):
    x = str(x).strip()          
    x = x.replace('%', '')       
    x = x.replace(',', '.')      
    try:
        return float(x)
    except ValueError:
        return np.nan            

In [85]:
df['salary_increase'] = df['salary_increase'].apply(clean_percentage)
df['occupation_rate'] = df['occupation_rate'].apply(clean_percentage)
df['unemployment_rate'] = df['unemployment_rate'].apply(clean_percentage)
df['informality_rate'] = df['informality_rate'].apply(clean_percentage)
df['quality_employment_probability'] = df['quality_employment_probability'].apply(clean_percentage)

In [86]:
cols_with_dollar

Index(['public_cost', 'private_cost', 'average_salary', 'women_salary',
       'men_salary', 'under_30_salary', 'over_30_salary', 'formal_salary',
       'informal_salary', 'postgrad_salary'],
      dtype='object')

In [87]:
def remove_zeros(x, mean_value):
    return mean_value if x == 0 else x

for col in cols_with_dollar:
    mean_value = df[col][df[col] != 0].mean()
    df[col] = df[col].apply(lambda x: remove_zeros(x, round(mean_value,2)))


In [88]:
df

,id,career,total_students,new_graduates_number,public_cost,private_cost,public_quality_rating,private_quality_rating,occupation_rate,unemployment_rate,...,career_rank,women_salary,men_salary,under_30_salary,over_30_salary,formal_salary,informal_salary,postgrad_percentage,postgrad_salary,salary_increase
0,0,administración_de_empresas,1300000,47537,50156.0,228756.00,Buena,Buena,96.8,3.3,...,18.0,19561.00,25207.00,16723.00,24358.00,23740.00,16242.00,8.5,43317.00,236.71
1,1,administración_en_el_ámbito_de_la_ingeniería_y...,79106,3539,42359.7,275512.22,-,-,94.7,5.3,...,NaN,18455.86,21249.23,16354.34,20990.76,21541.85,13974.02,6.0,34389.31,NaN
2,2,administración_pública,48254,1938,42359.7,275512.22,-,-,98.1,1.9,...,NaN,18455.86,21249.23,16354.34,20990.76,21541.85,13974.02,49.6,34389.31,NaN
3,3,administración_turística_y_de_restaurantes,61850,6029,44415.0,451397.00,Insegura,Muy Insegura,93.9,6.1,...,32.0,16651.00,20528.00,14702.00,19567.00,19757.00,12040.00,7.4,26889.00,174.49
4,4,adquisición_de_idiomas_extranjeros,77839,3083,47970.0,169694.00,Muy Insegura,Muy Insegura,94.1,5.9,...,40.0,16595.00,19242.00,14089.00,19676.00,18491.00,13691.00,3.0,27178.00,181.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,76,tecnología_para_la_protección_del_medio_ambiente,34168,3287,42359.7,275512.22,-,-,94.2,5.8,...,NaN,18455.86,21249.23,16354.34,20990.76,21541.85,13974.02,8.5,34389.31,NaN
77,77,terapia_y_rehabilitación,66243,8890,42359.7,275512.22,-,-,96.8,3.2,...,NaN,18455.86,21249.23,16354.34,20990.76,21541.85,13974.02,3.7,34389.31,NaN
78,78,trabajo_y_atención_social,157770,7898,33790.0,88790.00,Buena,Excelente,97.2,2.8,...,43.0,16935.00,17447.00,13446.00,18335.00,18159.00,12450.00,3.6,27548.00,188.47
79,79,vehículos_barcos_y_aeronaves_motorizadas,71470,6150,42359.7,275512.22,-,-,97.3,2.7,...,NaN,18455.86,21249.23,16354.34,20990.76,21541.85,13974.02,1.2,34389.31,NaN


In [89]:
df.describe()

,id,total_students,new_graduates_number,public_cost,private_cost,occupation_rate,unemployment_rate,informality_rate,quality_employment_probability,average_salary,career_rank,women_salary,men_salary,under_30_salary,over_30_salary,formal_salary,informal_salary,postgrad_percentage,postgrad_salary,salary_increase
count,81.000000,8.100000e+01,81.000000,81.000000,8.100000e+01,77.000000,77.000000,81.000000,71.000000,81.000000,59.000000,81.000000,81.000000,81.000000,81.000000,81.000000,81.000000,64.000000,81.000000,42.000000
mean,40.000000,2.227476e+05,8795.037037,42359.698765,2.755122e+05,96.466234,3.541558,29.333333,74.556338,19832.711358,19.322034,18455.859753,21249.228642,16354.339259,20990.761975,21541.848148,13974.017778,12.559375,34389.309753,194.052143
std,23.526581,2.750472e+05,12173.469877,9156.404714,1.724113e+05,1.828690,1.825575,12.166347,7.848780,4551.453166,13.379656,4047.690982,4832.172974,3284.042987,4918.939000,4686.748667,2673.869871,15.789898,6252.781390,32.972386
min,0.000000,2.111700e+04,0.000000,23152.000000,3.629100e+04,89.700000,0.100000,8.600000,51.300000,10130.000000,1.000000,10325.000000,5520.000000,10338.000000,9732.000000,11301.000000,8626.000000,0.200000,16390.000000,75.560000
25%,20.000000,6.420200e+04,1459.000000,40978.000000,1.762090e+05,95.700000,2.300000,21.300000,70.750000,17384.000000,8.000000,16651.000000,19174.000000,14384.000000,18901.000000,19292.000000,12355.000000,3.550000,32187.000000,177.017500
50%,40.000000,1.098960e+05,4347.000000,42359.700000,2.755122e+05,96.600000,3.400000,26.100000,74.600000,19832.710000,16.000000,18455.860000,21249.230000,16354.340000,20990.760000,21541.850000,13974.020000,7.300000,34389.310000,195.815000
75%,60.000000,2.518240e+05,10476.000000,46870.000000,2.755122e+05,97.700000,4.300000,36.500000,80.550000,22021.000000,30.500000,19942.000000,23349.000000,17393.000000,23412.000000,23525.000000,15000.000000,13.025000,34421.000000,216.765000
max,80.000000,1.300000e+06,70987.000000,70605.000000,1.200000e+06,99.900000,10.300000,77.500000,90.300000,43587.000000,45.000000,38077.000000,46811.000000,28167.000000,44690.000000,45034.000000,24471.000000,90.800000,56013.000000,247.520000


In [90]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 23 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              81 non-null     int64  
 1   career                          81 non-null     object 
 2   total_students                  81 non-null     int64  
 3   new_graduates_number            81 non-null     int64  
 4   public_cost                     81 non-null     float64
 5   private_cost                    81 non-null     float64
 6   public_quality_rating           81 non-null     object 
 7   private_quality_rating          81 non-null     object 
 8   occupation_rate                 77 non-null     float64
 9   unemployment_rate               77 non-null     float64
 10  informality_rate                81 non-null     float64
 11  quality_employment_probability  71 non-null     float64
 12  average_salary                  81 non

In [91]:
df.to_csv('data_all.csv')